# 4. The fruit fly brain as a credit model

## What you will learn

- What a connectome (a complete wiring diagram of a brain) is, and how to build a small network from the fly's.
- How a network whose wiring is **fixed** can still make predictions, when only a simple logistic readout (the last step,
  which turns the network's pattern into a probability) is trained.
- Why a **shuffled-wiring control** is needed before you say that "biology" helped, and what the lecture found.

## What you need

- The `model/` folder of this repository, and the Python environment from `model/requirements.txt`.
- An internet connection on the first run: it downloads about 133 MB (two public files of 100,804,642 and 31,720,298 bytes). The files
  are kept (**cached**) in the folder `.cache/tcf_credit/fly` inside your home folder, outside the repository, so later
  runs need no internet. Section 2 prints the exact folder.
- Notebooks 1 to 3 help but are not required. Everything here runs on **example data** (made up, see below), so
  the numbers mean nothing about real companies.
- Time: about 1 to 2 minutes after the first download (the slow whole-brain run is switched off, see section 8).

## 1. What is a connectome?

A **neuron** is a nerve cell. It receives signals from other neurons and sends signals on. A **synapse** is the
contact point where one neuron passes a signal to another. A **connectome** is the complete list of which neuron
connects to which, and how strongly: a wiring diagram of a whole brain, like a table of every road
between every pair of cities.

Researchers mapped the neurons in the brain of one adult fruit fly and the connections between them (how it was imaged and traced is described by
Dorkenwald et al. 2024; the citation is in `model/FLY-DATA.md`). The result is the **FlyWire** connectome (version 783,
about 139,000 neurons). Shiu et al. 2024 gave each connection a sign: a **positive** value means the
sending neuron *excites* the receiving one (pushes it to fire), a **negative** value means it *inhibits* it (holds it back).

We use this wiring as a ready-made network for credit scoring. Three rules keep the experiment clean:

1. **The wiring is fixed.** It comes from the animal, not from our data. We never adjust it, so it cannot be tuned to
   the training years.
2. **Only the readout learns.** A logistic regression (the same kind of model as the main model, the logit, from notebook 2) learns one weight per
   Kenyon cell (a memory-layer neuron of the fly's learning centre; section 4 shows them). That is the only thing fitted.
3. **We compare with shuffled wiring.** Same number of connections and same strengths, but connected to random
   targets. If the real fly wiring does not beat the shuffled wiring, we cannot credit the biology.

## 2. Download the wiring

Two public files are downloaded the first time (and checked against a fingerprint, so everyone uses exactly the same
wiring):

| File | Size | Holds |
|---|---|---|
| `Connectivity_783.parquet` | about 101 MB | every connection of the brain, with signed connection strengths |
| `Supplemental_file1_neuron_annotations.tsv` | about 32 MB | what each neuron is (its cell class) |

They are stored in a **cache** (a folder where downloaded files are kept so they are not downloaded again) outside the repository, so they are not mixed in with the project files. `model/FLY-DATA.md` gives the
sources, the licences and how to cite them. The code below prints where the cache is and loads the data. The first
run downloads about 133 MB and takes as long as that download takes on your connection; later runs read the cache and take a few seconds.

In [ ]:
import shutil
import sys
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", message="Sparse invariant checks")      # hides a PyTorch warning whose text starts with 'Sparse invariant checks'

# Find the package code and the example-data generator, whether Jupyter was started in model/notebooks, in model, or in the repository folder.
for base in (Path(".."), Path("model"), Path(".")):
    if (base / "synthetic" / "make_panel.py").exists():
        sys.path[:0] = [str((base / "synthetic").resolve()), str((base / "src").resolve())]
        break
from make_panel import make_panel

from tcf_credit import brain, evaluate, fly

print("Cache folder for the wiring files:", fly.fly_dir())

In [ ]:
t0 = time.time()
connections, annotations = fly.connectome()       # downloads on the first run, then reads the files
print(f"loaded in {time.time() - t0:.0f} seconds (the first run includes the download)")

## 3. How big is the fly's brain?

`connections` has one row per connected pair: `pre` (the sending neuron), `post` (the receiving neuron) and `weight`
(the signed connection strength from the Shiu et al. table: positive excites, negative inhibits). `annotations` says what each neuron is. The **cell class** is its family, for
example "Kenyon_Cell" or "ALPN".

In [ ]:
neurons = np.unique(np.concatenate([connections.pre.to_numpy(), connections.post.to_numpy()]))
print(f"neurons that appear in the wiring: {len(neurons):,}")
print(f"connections (pairs of neurons):    {len(connections):,}")
print(f"  exciting (weight > 0):           {(connections.weight > 0).mean():.0%}")
print(f"  inhibiting (weight < 0):         {(connections.weight < 0).mean():.0%}")
print(f"total connection strength:         {connections.weight.abs().sum():,.0f}")
print()
print("The ten biggest families of neurons (cell classes):")
print(annotations.cell_class.value_counts().head(10).to_string())

## 4. The mushroom body: the part we build first

The **mushroom body** is the fly's learning centre. It links smells to good or bad outcomes. Its wiring has two
layers:

- **Projection neurons** (cell class `ALPN`) carry smell information in. They are the *input* layer.
- **Kenyon cells** (cell class `Kenyon_Cell`) form the *memory* layer. There are many more of them than inputs, and each
  one listens to only a handful of inputs, apparently at random.

This looks like a lot of machine learning ideas in one animal: a few inputs are *expanded* onto many cells, and
only the most excited cells stay on, so at any moment few cells are on (a **sparse code**, as when a few lights in a big hall are lit). A later layer can then separate patterns that were hard to
separate before. In our model, the later layer is the logistic readout.

In [ ]:
edges = fly.mushroom_body_edges(connections, annotations)       # input neuron -> Kenyon cell, with signed weight
per_kenyon = edges.groupby("kc").size()
print(f"input neurons (ALPN) with a connection onto a Kenyon cell: {edges.pn.nunique()}")
print(f"Kenyon cells with an input:                                {edges.kc.nunique():,}")
print(f"connections between them:                                  {len(edges):,}")
print(f"exciting share:                                            {(edges.weight > 0).mean():.0%}")
print()
print("How many different inputs does one Kenyon cell listen to?")
counts = per_kenyon.value_counts().sort_index()
for n, k in counts.items():
    if n <= 12:
        print(f"  {n:2d} inputs: {k:5d} cells  " + "#" * int(60 * k / counts.max()))
print(f"  more than 12 inputs: {int(counts[counts.index > 12].sum())} cells")

Most Kenyon cells listen to 4 to 8 inputs; the typical one has 5 or 6. Now we turn this into a network. The class `MushroomBody`
does three things for each company:

1. Every input neuron is tied to **one** of the company's prepared inputs (one column of the logit's **design matrix**: the table of prepared numbers, one row per company and one column
   per input, a ratio or a rating flag, ...), and to its **high** or its **low** side. A neuron cannot go below zero, so a
   neuron on the high side fires when its ratio is high, and a neuron on the low side fires when it is low. The
   `seed` draws this assignment. A **random seed** is a starting number for the computer's random draws: the same seed
   always gives the same "random" choices, so a result can be repeated, and a different seed gives a different draw.
   The assignment is fixed before any outcome is seen.
2. Each Kenyon cell adds up what its inputs send, weighted by the connection strengths of the real wiring.
3. Only the most driven **5%** of Kenyon cells stay on. The rest are silent.

In [ ]:
mushroom_body = fly.MushroomBody("real", seed=1, C=0.01)        # C: the readout's penalty; smaller C holds its weights closer to zero (guards against over-fitting)
print("wiring (input neurons x Kenyon cells):", mushroom_body.wiring.shape)
print("Kenyon cells that stay on per company: ", mushroom_body.active, f"({fly.ACTIVE_SHARE:.0%})")

## 5. Example data

Before we can send a company through the network we need companies. The **example data** is a made-up table of
company-years with the same columns as the real panel (except the licensed stock-database identifier), produced by `model/synthetic/make_panel.py`. We write it to a
temporary folder, read it back, and delete the folder at the end of the notebook. The rule behind the made-up
downgrades is simple, so the models can find something, but **nothing here says anything about real companies**.

We train on early years, leave one year out and test on later years, as in notebook 2. A report counts as followed by
a downgrade if one comes within 12 months of it, so a gap year keeps the answers of the last training reports from
overlapping the test years.

The example data is first **prepared**: very large or very small values are cut back to the 1st and 99th percentile
of the training years (**winsorising**, so a few extreme companies do not dominate), gaps are filled with the typical
value, and every column is scaled so that 0 is typical. All of this is learned on the training years only, so nothing
from the test years leaks into the model.

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="fly_notebook_"))
make_panel(n_firms=400, seed=1).to_csv(workdir / "example_panel.csv", index=False)
panel = pd.read_csv(workdir / "example_panel.csv")
panel = panel[panel.has_prices == 1]                              # the models use only rows with stock-market inputs

inputs = evaluate.RATIOS + evaluate.RATING + evaluate.BINARY + evaluate.MARKET
train = panel[panel.fy_end_year <= 2019]
test = panel[panel.fy_end_year.between(2021, 2023)]               # 2020 is the gap year

prep = evaluate.Prep(inputs).fit(train)                           # cut extremes, fill gaps, scale: learned on training years only
x_train, x_test = prep.transform(train), prep.transform(test)
y_train, y_test = train.label.to_numpy(), test.label.to_numpy()
print(f"training rows: {len(train):,} (downgrade share {y_train.mean():.1%});  test rows: {len(test):,} ({y_test.mean():.1%})")
print(f"columns after preparation: {x_train.shape[1]}")

## 6. One company going in

Take the first test company. Its prepared row has one number per input column (scaled so that 0 is typical).
We assign the input neurons, drive the network and see which Kenyon cells turn on.

In [ ]:
mushroom_body.assign_inputs(x_train.shape[1])            # fit() does this too, with the same seed, so the result is the same
company = test.iloc[[0]]
row = prep.transform(company)

print("Prepared inputs of this company (first 8 columns):")
names = inputs + ["(missing flags, sector flags ...)"]
for name, value in zip(names[:8], row[0, :8]):
    print(f"  {name:14s} {value:+.2f}")

active = mushroom_body.active_cells(row).numpy()[0]
print()
print(f"{mushroom_body.wiring.shape[1]:,} Kenyon cells; these {len(active)} are on (their numbers):")
print(sorted(active.tolist())[:20], "...")

Two companies with similar inputs should turn on more of the same cells than two companies with different inputs. We check this with the share of cells that two companies have in common.

In [ ]:
def overlap(a, b):
    a, b = set(a.tolist()), set(b.tolist())
    return len(a & b) / len(a | b)

cells = mushroom_body.active_cells(prep.transform(test.iloc[:200])).numpy()
rng = np.random.default_rng(0)
pairs = [rng.choice(200, 2, replace=False) for _ in range(300)]
share = np.array([overlap(cells[i], cells[j]) for i, j in pairs])
closeness = np.array([np.linalg.norm(prep.transform(test.iloc[[i]]) - prep.transform(test.iloc[[j]])) for i, j in pairs])
close = closeness <= np.median(closeness)
print(f"cells in common, pairs of similar companies:   {share[close].mean():.0%}")
print(f"cells in common, pairs of different companies: {share[~close].mean():.0%}")

Similar companies share more cells. That is why a readout that only sees "which cells are on" can still tell
companies apart: the fixed network is a random-looking, sparse re-coding of the inputs.

## 7. Train the readout and compare

Now `fit` ties the inputs to the network (as above) and trains the readout. Only the weights of the readout change;
the wiring stays as it is. We compare:

- the **logit**, an ordinary logistic regression on the prepared inputs (our main model);
- the fly **mushroom body with real wiring**;
- the fly mushroom body with **shuffled wiring**: same number of connections and strengths, random targets.

Each fly model is run with three seeds (three different input assignments and shuffles), because one draw could be
lucky or unlucky. The score is the **ROC-AUC** (the chance that a company with a later downgrade gets a higher score
than one without; 0.5 is chance, 1.0 is perfect). Settings (`C`) are chosen by hand here; in the lecture they were
chosen on **validation years** (years held back before the test years, used only to choose settings).

In [ ]:
t0 = time.time()
results = {}
logit = LogisticRegression(C=0.1, max_iter=5000).fit(x_train, y_train)
results["logit"] = [roc_auc_score(y_test, logit.predict_proba(x_test)[:, 1])]
for wiring in ("real", "shuffled"):
    results[f"fly {wiring} wiring"] = []
    for seed in (1, 2, 3):
        model = fly.MushroomBody(wiring, seed=seed, C=0.01).fit(x_train, y_train)
        results[f"fly {wiring} wiring"].append(roc_auc_score(y_test, model.predict_proba(x_test)[:, 1]))

table = pd.DataFrame({name: {"mean AUC": np.mean(v), "lowest": np.min(v), "highest": np.max(v)}
                      for name, v in results.items()}).T
print(table.round(3).to_string())
real, shuffled = np.array(results["fly real wiring"]), np.array(results["fly shuffled wiring"])
print(f"\nreal minus shuffled, same seed: {np.round(real - shuffled, 3).tolist()}")
print(f"({time.time() - t0:.0f} seconds)")

How to read this table:

- The three seeds of one fly model move by about a hundredth of AUC or more, and the test set is small. A difference
  between the real and the shuffled line of that size is **noise**.
- Look at the last line: if "real minus shuffled" is sometimes positive and sometimes negative, the real wiring has no
  consistent edge.
- The logit and the fly models see the same inputs. The fly adds a fixed, random-looking expansion, and its only
  trained part is a logistic regression again.

## 8. The whole brain: spreading activity first

`brain.py` goes one step further. In the real animal, a smell reaches the Kenyon cells only after the sensory neurons and the
antennal lobe have processed it. `WholeBrain` lets the activity spread through the wiring in a similar way: the company's inputs start the activity,
and then it spreads along all 15 million connections for a few steps. At each step a neuron adds what its exciting
inputs send, subtracts what its inhibiting inputs send, and cannot go below zero. After the last step, the most
driven 5% of Kenyon cells stay on and the same logistic readout reads them.

The idea in a tiny network of five neurons (neurons 0 and 1 are inputs; neuron 4 is the "Kenyon cell"):

In [ ]:
# rows: sender, columns: receiver. Positive = excites, negative = inhibits.
wiring = np.array([[0, 0, 2, 1, 0],
                   [0, 0, 0, -3, 0],
                   [0, 0, 0, 0, 2],
                   [0, 0, 0, 0, 1],
                   [0, 0, 0, 0, 0]], dtype=float)
state = np.array([1.0, 1.0, 0, 0, 0])                    # step 0: both inputs are active
print("step 0:", state)
for step in (1, 2):
    state = np.maximum(wiring.T @ state, 0)               # add excitation, subtract inhibition, never below zero
    state[:2] = 1.0                                       # the inputs keep receiving the company's values
    print(f"step {step}:", state)

Neuron 3 receives +1 from neuron 0 and -3 from neuron 1, so it is pushed below zero and stays silent. Only the
path through neuron 2 reaches neuron 4. The real network does this for 139,000 neurons at once.

The next cell fits the same `WholeBrain` class on a small made-up network (so it runs in seconds), only to show
that it is used like any other model: `fit` and `predict_proba`.

In [ ]:
import scipy.sparse as sp
rng = np.random.default_rng(0)
small = sp.random(120, 120, density=0.05, random_state=0, format="csr", dtype=np.float32)
small.data = np.round(small.data * 20 - 6).astype(np.float32)          # some connections inhibit
small_graph = brain.Graph(small, inputs=np.arange(10), cells=np.arange(80, 120))
toy_brain = brain.WholeBrain(small_graph, seed=1, C=0.01, steps=3).fit(x_train, y_train)
print("AUC of the toy network (it has no meaning, it only shows the steps run):",
      round(roc_auc_score(y_test, toy_brain.predict_proba(x_test)[:, 1]), 3))

**The real whole brain is slow.** Building the 139,000-neuron network and fitting two models (real and shuffled)
is much slower than the rest of this notebook (in the lecture's real-data run a whole-brain model took 220 to 1,831 seconds,
against 89 to 182 for a mushroom-body model; it was not timed on the example data), so it is switched off to keep this notebook
short. Set `RUN_WHOLE_BRAIN = True` to run it with one seed; it prints one AUC line for the real wiring and one for the shuffled wiring.

In [ ]:
RUN_WHOLE_BRAIN = False                                  # True: much slower (lecture real-data run: 220 to 1,831 s per whole-brain model; not timed on example data)

if RUN_WHOLE_BRAIN:
    t0 = time.time()
    for wiring in ("real", "shuffled"):
        whole = brain.WholeBrain(wiring, seed=1, C=0.01).fit(x_train, y_train)
        print(f"whole brain, {wiring:8s} wiring: AUC {roc_auc_score(y_test, whole.predict_proba(x_test)[:, 1]):.3f}")
    print(f"({time.time() - t0:.0f} seconds)")
else:
    print("Whole brain skipped (RUN_WHOLE_BRAIN = False).")

## 9. What the lecture found, and what it means

On the lecture's **real data** (test years 2016 to 2024, 6,380 annual reports, 862 followed by a downgrade) the fly
models scored a ROC-AUC between **0.759 and 0.807**. The logit scored **0.802**; the single flexible
engines scored 0.787 to 0.795. The highest fly score, 0.807, came from a whole-brain model with *shuffled* wiring. On PR-AUC (the second quality number from notebook 2), which
looks at the top of the list where the flags are, every fly model scored lower than the logit: **0.369 to 0.476, against
0.494**. Across the three seeds, the ROC-AUC of one fly model moved by about **0.025 to 0.036**. And the **real fly wiring had
no consistent edge over shuffled wiring**.

What this means, carefully:

- **On this data, the fly models were not better than the logit.** The best of them came close to it on ROC-AUC, but some runs were
  clearly below it (the weakest, 0.759), and every fly model was lower on PR-AUC.
- **The real wiring gave no consistent edge over the shuffled wiring.** If the particular pattern of the fly's
  connections helped here, real wiring would beat shuffled wiring in most seeds. It did not: it was ahead in
  some seeds and behind in others.
- **We did not find out what does the work in these models.** We do not claim that the design (spreading the inputs onto
  many cells, with only a few on at a time) or the biology is what works. No run took the expansion away, and the plain
  logit scored about the same on ROC-AUC and higher on PR-AUC. What the shuffled control does show is that the particular
  wiring was not needed for the scores the fly models reached. That is a statement about this experiment. It is not a
  finding that the fly brain "knows nothing" about credit, and it does not show that biology can never help.
- **The shuffled control is what lets us say this.** Without it, a good fly score could have been mistaken for a
  discovery about biology; the best fly score here came from shuffled wiring.
- The logit stays the main model: it is simpler, explainable, its ROC-AUC is about the same as the best fly models' and its
  PR-AUC is higher.

### Why does this not prove the fly brain knows nothing?

Because the experiment is narrow:

- **Sample size.** There are 862 downgrades in the lecture's test years, and a few seeds per model. Differences of a
  hundredth of AUC are inside the noise, so a small real edge could be hidden.
- **One task.** We asked one question, whether S&P downgrades a company within 12 months, from yearly report numbers, the S&P rating state, stock-market numbers and the economy (the small example in this notebook uses fewer inputs).
  A different task could behave differently.
- **One wiring version.** We used one fly's connectome (FlyWire version 783), in one mushroom-body setup, with one
  way of feeding company numbers into it. Other choices could change the result.
- **The readout is the only learner.** The wiring is never trained, so it could only help if a fixed random-looking
  pattern happened to suit the task. A result of "no edge" says how this simple use of the wiring behaved, not
  what the fly's brain contains.

What we can say is narrow too: here, real wiring was no better than shuffled wiring.

## 10. Tidy up

The example data lived in a temporary folder. Delete it. (The downloaded wiring stays in its cache folder, so the next
run is fast; you can delete that folder too, see `model/FLY-DATA.md`.)

In [ ]:
shutil.rmtree(workdir)
print("temporary folder removed:", not workdir.exists())

## Check yourself

1. What is trained in the fly model, and what is not?
2. Why do we also build a model with shuffled wiring?
3. If real wiring beats shuffled wiring by 0.005 AUC in one seed, would you say the fly's biology helps? Why?
4. Why does the lecture result not prove that the fly brain "knows nothing" about credit?

<details>
<summary>Answers</summary>

1. Only the logistic readout (one weight per Kenyon cell and a constant) is trained. The wiring, which input neuron
   listens to which ratio, and the rule "the strongest 5% of cells stay on" are fixed before the outcomes are seen.
2. It keeps the number of connections and their strengths but destroys who connects to whom. If the real wiring does
   not beat it consistently, there is no sign that the real pattern of connections helps.
3. No. The three seeds of one model already move by more than that (about 0.025 to 0.036 AUC on the lecture's real data), and the test set is small. You would need
   an edge that shows up in most seeds and most test years.
4. It is narrow: a small sample (862 downgrades), one task, one wiring version, and the readout is the only part that
   learns. It shows that real wiring was no better than shuffled wiring here, not that the fly brain "knows nothing".
</details>